# ProboSed 01 — JCORES VCD Mining and MTD Catalog
**IODP Expedition 405 — Site C0019J, Japan Trench**

Extracts stability scores from the JCORES VCD PDF and produces the MTD boundary catalog.

### CORC0019.pdf page format:
```
Hole C0019J Core 7K, interval 139 to 142.805 m (core depth below seafloor)
```
Core ID extracted from `Core 7K`, depth from `interval 139 to ...`

### Output files:
- `C0019J_VCD_stability_log.csv`
- `C0019J_MTD_catalog.csv`
- `C0019J_stability_profile.png`

In [ ]:
# Cell 1 — Install dependencies
!pip install pymupdf openpyxl pandas numpy matplotlib --quiet

In [ ]:
# Cell 2 — Mount Google Drive and clone ProboSed
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys, os, shutil

repo_path = '/content/ProboSed'
if os.path.exists(repo_path):
    shutil.rmtree(repo_path)

result = subprocess.run(
    ['git', 'clone', 'https://github.com/rocknrene/ProboSed.git', repo_path],
    capture_output=True, text=True
)
print('Clone return code:', result.returncode)
if result.returncode != 0:
    print('STDERR:', result.stderr)

# Add to path — do this here so all subsequent cells can import
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

print('ProboSed contents:', os.listdir(repo_path))
print('core_ml exists:', os.path.exists(os.path.join(repo_path, 'core_ml')))

In [ ]:
# Cell 3 — Verify import and check lexicon
import sys
if '/content/ProboSed' not in sys.path:
    sys.path.insert(0, '/content/ProboSed')

from core_ml.labeler import JCORESMiner, JCORES_LEXICON, DRILLING_CAUSE_TERMS

print('Lexicon checks:')
print(f'  minor in lexicon:          {"minor" in JCORES_LEXICON}  (should be False)')
print(f'  disturb in lexicon:        {"disturb" in JCORES_LEXICON}  (should be False)')
print(f'  sheared in lexicon:        {"sheared" in JCORES_LEXICON}  (should be True)')
print(f'  high disturbance:          {"high disturbance" in JCORES_LEXICON}  (should be True)')
print(f'  due to drilling in causes: {"due to drilling" in DRILLING_CAUSE_TERMS}  (should be True)')
print(f'\nCORE_PATTERN:  {JCORESMiner.CORE_PATTERN.pattern}')
print(f'DEPTH_PATTERN: {JCORESMiner.DEPTH_PATTERN.pattern}')

In [ ]:
# Cell 4 — Configure paths
import os

DATA_DIR   = '/content/drive/MyDrive/iodp/X405/Data & Data Tracking'
OUTPUT_DIR = '/content/drive/MyDrive/iodp/X405/ProboSed_Output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

VCD_PDF      = os.path.join(DATA_DIR, 'vcds', 'CORC0019.pdf')
SUMMARY_XLSX = os.path.join(DATA_DIR, '405-C0019J_CoreSummary_2410280924.xlsx')
STABILITY_CSV = os.path.join(OUTPUT_DIR, 'C0019J_VCD_stability_log.csv')
MTD_CSV       = os.path.join(OUTPUT_DIR, 'C0019J_MTD_catalog.csv')
PROFILE_PNG   = os.path.join(OUTPUT_DIR, 'C0019J_stability_profile.png')

for path, label in [(VCD_PDF, 'VCD PDF'), (SUMMARY_XLSX, 'Core summary Excel')]:
    status = 'FOUND' if os.path.exists(path) else 'NOT FOUND — check path'
    print(f'{label}: {status}')
    print(f'  {path}')

In [ ]:
# Cell 5 — Verify PDF text format
import sys
if '/content/ProboSed' not in sys.path:
    sys.path.insert(0, '/content/ProboSed')

import fitz
from core_ml.labeler import JCORESMiner

doc = fitz.open(VCD_PDF)
print(f'PDF pages: {len(doc)}')
print()

for i in range(min(3, len(doc))):
    page = doc.load_page(i)
    text = page.get_text()
    lines = [l.strip() for l in text.split('\n') if l.strip()][:4]
    print(f'Page {i+1} header:')
    for l in lines:
        print(f'  {l}')
    core_m  = JCORESMiner.CORE_PATTERN.search(text)
    depth_m = JCORESMiner.DEPTH_PATTERN.search(text)
    core_id = (core_m.group(1) or core_m.group(2)).upper() if core_m else None
    depth   = float(depth_m.group(1)) if depth_m else None
    print(f'  -> Core={core_id}  Depth={depth}')
    print()

doc.close()

if core_id is None:
    print('WARNING: Core ID not found. Check VCD_PDF path.')
else:
    print('Core ID and depth extraction working correctly.')

In [ ]:
# Cell 6 — Build depth backbone
import sys
if '/content/ProboSed' not in sys.path:
    sys.path.insert(0, '/content/ProboSed')

from core_ml.labeler import JCORESMiner

miner    = JCORESMiner(VCD_PDF, SUMMARY_XLSX)
backbone = miner.build_backbone()

print(f'Backbone sample (first 10 rows):')
print(backbone.head(10).to_string(index=False))

In [ ]:
# Cell 7 — Fix lexicon and extract stability scores
#
# Lexicon fixes applied here based on spot-check validation results:
#   REMOVED: 'disturb', 'disturbed' — too generic, fired on every page
#   REMOVED: 'color banding', 'colour banding' — fired on lithology headers
#   REMOVED: 'soft deformation', 'soft sediment deform' — false positives
#   ADDED:   'shear deformation', 'evidence of shear' — found in core 87K text
#   ADDED:   'soft-sediment deform', 'some evidence of deform' — section-level text
#
# These changes are applied in-memory. Update labeler.py on GitHub to make permanent.

import sys
if '/content/ProboSed' not in sys.path:
    sys.path.insert(0, '/content/ProboSed')

from core_ml import labeler as lm

# Remove false positive terms
for term in ['disturb', 'disturbed', 'color banding',
             'colour banding', 'soft deformation',
             'soft sediment deform']:
    if term in lm.JCORES_LEXICON:
        del lm.JCORES_LEXICON[term]
        print(f'Removed: "{term}"')

# Add terms found during spot-check
lm.JCORES_LEXICON['shear deformation']      = 1
lm.JCORES_LEXICON['evidence of shear']      = 1
lm.JCORES_LEXICON['soft-sediment deform']   = 1
lm.JCORES_LEXICON['some evidence of deform']= 1
print('Added: shear deformation variants')
print()
print('Active score-0 terms:', [k for k,v in lm.JCORES_LEXICON.items() if v==0])
print('Active score-1 terms:', [k for k,v in lm.JCORES_LEXICON.items() if v==1])
print()

# Extract with fixed lexicon
vcd_df = miner.extract(backbone)

print(f'\nStability score distribution:')
print(vcd_df['Stability'].value_counts().sort_index())
print(f'\nDepth-matched: {vcd_df["Depth_m"].notna().sum()} / {len(vcd_df)}')
print(f'Drilling artifacts: {vcd_df["Drilling_Artifact"].sum()}')
print(f'\nSpot-check cores:')
for core in ['1K', '14K', '40K', '50K', '60K', '75K', '87K']:
    rows = vcd_df[vcd_df['Core'] == core]
    if not rows.empty:
        r = rows.iloc[0]
        print(f'  {core}: score={r["Stability"]}  '
              f'term="{r["Disturbance"]}"  '
              f'drilling={r["Drilling_Artifact"]}')

In [ ]:
# Cell 8 — Save stability log
vcd_df.to_csv(STABILITY_CSV, index=False)
print(f'Saved: {STABILITY_CSV}')
print(f'Rows: {len(vcd_df)}')

In [ ]:
# Cell 9 — Build MTD catalog
# Drilling artifact intervals excluded before catalog construction.
import sys
if '/content/ProboSed' not in sys.path:
    sys.path.insert(0, '/content/ProboSed')

from core_ml.labeler import JCORESMiner

geo_df      = vcd_df[~vcd_df['Drilling_Artifact']].copy()
mtd_catalog = JCORESMiner.score_to_mtd_catalog(geo_df, stability_threshold=1)

mtd_catalog.to_csv(MTD_CSV, index=False)
print(f'MTD intervals identified: {len(mtd_catalog)}')
print(f'Saved: {MTD_CSV}')
print()
if len(mtd_catalog) > 0:
    print(mtd_catalog.to_string(index=False))

In [ ]:
# Cell 10 — Plot stability profile
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

plot_df = vcd_df[~vcd_df['Drilling_Artifact']].dropna(
    subset=['Depth_m','Stability']
).sort_values('Depth_m')

fig, ax = plt.subplots(figsize=(6, 16))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f7f7f7')

ax.step(plot_df['Stability'], plot_df['Depth_m'],
        where='post', color='#2c3e50', lw=1.5, zorder=3)

for _, row in mtd_catalog.iterrows():
    ax.axhspan(row['top_m'], row['bottom_m'],
               alpha=0.2, color='#cc2222', zorder=1)

ax.axhline(820, color='#ff4444', ls='--', lw=1.5, alpha=0.8,
           label='Plate boundary fault (~820 mbsf)')

ax.invert_yaxis()
ax.set_xlim(-0.5, 3.5)
ax.set_xticks([0, 1, 2, 3])
ax.set_xticklabels(
    ['Slurried\n(0)', 'Scaly\n(1)', 'Coherent\n(2)', 'Intact\n(3)'],
    fontsize=10
)
ax.set_ylabel('Depth (m CSF-A)', fontsize=12)
ax.set_title('C0019J Stability Profile\n(JCORESMiner extraction)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='x', alpha=0.3)
ax.text(0.02, 0.02, f'MTD intervals: {len(mtd_catalog)}',
        transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle='round', facecolor='#cc2222', alpha=0.2))

plt.tight_layout()
plt.savefig(PROFILE_PNG, dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {PROFILE_PNG}')

In [ ]:
# Cell 11 — Summary
print('=== MTD CATALOG SUMMARY ===')
print(f'Total MTD intervals:          {len(mtd_catalog)}')
if len(mtd_catalog) > 0:
    print(f'Shallowest top:               {mtd_catalog["top_m"].min():.1f} m CSF-A')
    print(f'Deepest base:                 {mtd_catalog["bottom_m"].max():.1f} m CSF-A')
    print(f'Mean thickness:               {mtd_catalog["thickness_m"].mean():.1f} m')
    print(f'Total MTD thickness:          {mtd_catalog["thickness_m"].sum():.1f} m')
    print()
    print('Individual intervals:')
    for _, row in mtd_catalog.iterrows():
        print(f'  {row["mtd_id"]}: {row["top_m"]:.1f} - '
              f'{row["bottom_m"]:.1f} m  '
              f'({row["thickness_m"]:.1f} m thick, '
              f'mean stability = {row["mean_stability"]:.1f})')
print()
print(f'Drilling artifacts excluded:   {vcd_df["Drilling_Artifact"].sum()}')
print(f'Depth-unmatched pages:         {vcd_df["Depth_m"].isna().sum()}')
print()
print('Output files:')
print(f'  {STABILITY_CSV}')
print(f'  {MTD_CSV}')
print(f'  {PROFILE_PNG}')
print()
print('Next: open notebook 07 (spot-check), reload Cell 3, rerun Cells 4 and 5.')